[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ronsong1234/GrandQC_IDC-Validation/blob/main/notebooks/03_ybr_ict_prevalence.ipynb)


# 03 — YBR_ICT Prevalence in IDC (OpenSlide-decoding safety)

**Purpose.** Measure how many TCGA-BRCA slides in IDC are JPEG 2000 **YBR_ICT** — the photometric interpretation OpenSlide 4.0.1 silently mis-decodes. This is the exposure of any OpenSlide-based pipeline run on IDC pathology data.

**Inputs.** IDC index (via `idc-index`); public `idc-open-data` S3 bucket for header reads.

**Expected outputs.** `results/summary_tables/ybr_ict_prevalence.csv` and a printed count.

**Environment.** `environment.yml` / `requirements-lock.txt`. **Runtime.** ~3-6 min (39 header reads over HTTP).
**GPU.** not required.


### Why headers, not the index

The IDC index exposes `transfer_syntax_name` (JPEG vs JPEG 2000) and `Manufacturer`, but **not** `PhotometricInterpretation`. YBR_ICT is a photometric *within* JPEG 2000 — 5 of the 39 BRCA J2K slides decode as plain RGB and are unaffected — so JPEG 2000 alone is not a proxy. We read the photometric from ~300 KB of one instance per series via HTTP range requests.

In [ ]:
# --- Colab bootstrap: clone this repo for src/, install deps, set path ---
import sys, os, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("GrandQC_IDC-Validation"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/ronsong1234/GrandQC_IDC-Validation.git"], check=True)
    sys.path.insert(0, "GrandQC_IDC-Validation/src")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "idc-index", "pydicom"], check=False)
else:
    sys.path.insert(0, os.path.abspath("../src"))
from inspect_dicom_headers import photometric_for_series
from idc_index import IDCClient
import pandas as pd, csv

c = IDCClient(); c.fetch_index('sm_index')
j2k = c.sql_query("""
    SELECT s.ContainerIdentifier AS bc, i.crdc_series_uuid AS crdc,
           s.min_PixelSpacing_2sf AS mm
    FROM index i JOIN sm_index s ON i.SeriesInstanceUID = s.SeriesInstanceUID
    WHERE i.collection_id = 'tcga_brca'
      AND i.transfer_syntax_name LIKE '%2000%'
      AND array_to_string(s.staining_usingSubstance_CodeMeaning, ', ') LIKE '%hematoxylin%'
      AND s.ContainerIdentifier LIKE '%-DX%'
""")
print(f'{len(j2k)} JPEG 2000 BRCA-DX slides to inspect')

In [ ]:
results = {}
for _, r in j2k.iterrows():
    try:
        results[r.bc] = photometric_for_series(r.crdc)
    except Exception as exc:
        results[r.bc] = f'ERR:{type(exc).__name__}'
    print(f'  {r.bc}: {results[r.bc]}')

In [ ]:
from collections import Counter
ybr = [b for b, p in results.items() if 'YBR' in p]
n_total = 1104  # BRCA DX H&E slides with a Zenodo mask
print('photometric tally:', dict(Counter(results.values())))
print(f'YBR_ICT (OpenSlide-vulnerable): {len(ybr)}/{n_total} = {100*len(ybr)/n_total:.1f}%')
print('by source site:', dict(Counter(b.split("-")[1] for b in ybr)))

os.makedirs('../results/summary_tables', exist_ok=True)
with open('../results/summary_tables/ybr_ict_prevalence.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['barcode', 'photometric_interpretation'])
    w.writerows(sorted(results.items()))
print('wrote ../results/summary_tables/ybr_ict_prevalence.csv')

**Result (IDC v24):** 29/1104 confirmed YBR_ICT (2.6%; up to 34/3.1% counting header-read failures), spanning sites AC (40x) and AO (20x). See `docs/findings.md` §2 and `docs/limitations.md` §5 (prevalence is cohort-specific).